In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

print("Snowflake connection successful!")
print("Database:", session.get_current_database())
print("Schema:", session.get_current_schema())

In [ ]:
# Load the training and testing tables

train_df = session.table("INSURANCE_ML.ML_PIPE.INSURANCE_TRAIN")
test_df = session.table("INSURANCE_ML.ML_PIPE.INSURANCE_TEST")

print("Training rows:", train_df.count())
print("Testing rows:", test_df.count())

In [ ]:
# Display a few training records

train_df.show(5)

In [ ]:
import sklearn
import xgboost

print("scikit-learn version:", sklearn.__version__)
print("XGBoost version:", xgboost.__version__)

In [ ]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

# Bring Snowflake data into pandas
train_pd = train_df.to_pandas()
test_pd = test_df.to_pandas()

# Separate features and target
X_train = train_pd.drop("CHARGES", axis=1)
y_train = train_pd["CHARGES"]

X_test = test_pd.drop("CHARGES", axis=1)
y_test = test_pd["CHARGES"]

# Define column types
numeric_features = ["AGE", "BMI", "CHILDREN"]
categorical_features = ["SEX", "SMOKER", "REGION"]

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_features
        )
    ]
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Preprocessing pipeline created successfully!")

In [ ]:
from xgboost import XGBRegressor

# XGBoost regression model
xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42
)

# Complete ML pipeline
model_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", xgb_model)
    ]
)

# Train the model
model_pipeline.fit(X_train, y_train)

print("XGBoost model trained successfully!")

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Make predictions on unseen test data
y_pred = model_pipeline.predict(X_test)

# Calculate evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Model Evaluation")
print("-----------------")
print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"R²   : {r2:.4f}")

In [ ]:
from snowflake.ml.registry import Registry

print("Snowflake Model Registry is available!")

In [ ]:
from snowflake.ml.registry import Registry

# Open the Model Registry in our existing database/schema
registry = Registry(
    session=session,
    database_name="INSURANCE_ML",
    schema_name="ML_PIPE"
)

print("Model Registry opened successfully!")

In [ ]:
model_ref = registry.log_model(
    model=model_pipeline,
    model_name="INSURANCE_CHARGES_MODEL",
    version_name="V1",
    sample_input_data=X_train,
    metrics={
        "MAE": float(mae),
        "RMSE": float(rmse),
        "R2": float(r2)
    },
    comment="XGBoost regression model for predicting insurance charges."
)

print("Model registered successfully!")
print("Model: INSURANCE_CHARGES_MODEL")
print("Version: V1")

In [ ]:
models = registry.show_models()

models

In [ ]:
model = registry.get_model("INSURANCE_CHARGES_MODEL")

versions = model.show_versions()

versions

In [ ]:
# Get the registered model version
model = registry.get_model("INSURANCE_CHARGES_MODEL")
model_v1 = model.version("V1")

print("Registered model version loaded:")
print(model_v1)

In [ ]:
# Run inference using the registered model
registered_predictions = model_v1.run(
    X_test,
    function_name="predict"
)

registered_predictions.head()

In [ ]:
from snowflake.ml.model import target_platform

model_ref_v2 = registry.log_model(
    model=model_pipeline,
    model_name="INSURANCE_CHARGES_MODEL",
    version_name="V2",
    sample_input_data=X_train,
    metrics={
        "MAE": float(mae),
        "RMSE": float(rmse),
        "R2": float(r2)
    },
    target_platforms=target_platform.WAREHOUSE_ONLY,
    pip_requirements=[
        "scikit-learn",
        "xgboost"
    ],
    artifact_repository_map={
        "pip": "snowflake.snowpark.pypi_shared_repository"
    },
    comment="XGBoost regression model for predicting insurance charges - Warehouse inference enabled."
)

print("Model V2 registered successfully!")
print("Warehouse inference enabled.")

In [ ]:
session.sql("USE DATABASE INSURANCE_ML").collect()
session.sql("USE SCHEMA ML_PIPE").collect()

print("Database:", session.get_current_database())
print("Schema:", session.get_current_schema())

In [ ]:
print(session.sql("SELECT CURRENT_DATABASE(), CURRENT_SCHEMA()").collect())

In [ ]:
from snowflake.ml.model import target_platform

model_ref_v2 = registry.log_model(
    model=model_pipeline,
    model_name="INSURANCE_CHARGES_MODEL",
    version_name="V2",
    sample_input_data=X_train,
    metrics={
        "MAE": float(mae),
        "RMSE": float(rmse),
        "R2": float(r2)
    },
    target_platforms=target_platform.WAREHOUSE_ONLY,
    pip_requirements=[
        "scikit-learn",
        "xgboost"
    ],
    artifact_repository_map={
        "pip": "snowflake.snowpark.pypi_shared_repository"
    },
    comment="XGBoost regression model for predicting insurance charges - Warehouse inference enabled."
)

print("Model V2 registered successfully!")
print("Warehouse inference enabled.")

In [ ]:
model = registry.get_model("INSURANCE_CHARGES_MODEL")
model_v2 = model.version("V2")

print("Registered model version loaded successfully!")
print("Model:", "INSURANCE_CHARGES_MODEL")
print("Version:", "V2")

In [ ]:
registered_predictions = model_v2.run(
    X_test,
    function_name="predict"
)

print("Inference completed successfully!")
registered_predictions.show()

In [ ]:
registered_predictions.head(10)

In [ ]:
print("Prediction type:", type(registered_predictions))
print("Columns:")
print(registered_predictions.columns.tolist())

In [ ]:
prediction_results = X_test.copy()

prediction_results["ACTUAL_CHARGES"] = y_test.values
prediction_results["PREDICTED_CHARGES"] = registered_predictions["output_feature_0"].values

prediction_results.head(10)

In [ ]:
print("Prediction rows:", len(prediction_results))
print("Prediction columns:", prediction_results.columns.tolist())

In [ ]:
prediction_table = "INSURANCE_PREDICTIONS"

session.write_pandas(
    prediction_results,
    table_name=prediction_table,
    database="INSURANCE_ML",
    schema="ML_PIPE",
    auto_create_table=True,
    overwrite=True
)

print("Prediction table created successfully!")
print("Table:", "INSURANCE_ML.ML_PIPE.INSURANCE_PREDICTIONS")

In [ ]:
prediction_df = session.table(
    "INSURANCE_ML.ML_PIPE.INSURANCE_PREDICTIONS"
)

print("Rows:", prediction_df.count())
prediction_df.show(10)